In [65]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

- Data: https://www.kaggle.com/datasets/harshlakhani2005/ai-tool-directory-2026-10000-real-world-tools
- Part 1 Notebook: https://www.kaggle.com/code/rudraprasadbhuyan/ai-tool-directory-2026-eda

In [66]:
path = r"C:\Users\Rudra\Desktop\kaggle\ai-tools\AI_Landscape_10k_Tools_2026.csv"

In [67]:
df = pd.read_csv(path)

In [68]:
df.sample(3)

,AI_Name,Developer,Release_Year,Intelligence_Type,Primary_Domain,Key_Functionality,Pricing_Model,API_Availability,Context_Window,Accessibility,Popularity_Votes,Website_URL
4187,Antipiracy,Quantum Creative,2026,Multimodal Generative AI,General/Other,AI-powered anti-piracy solution for protecting...,Freemium,Enterprise Only,8k tokens,Web App,30325,https://preview--anti-piracy-sphere.lovable.app
4374,Promptmule,Deep Intelligence,2023,Multimodal Generative AI,Coding,API cache-as-a-service for GenAI app developer...,Usage-Based,No,128k tokens,API / Developer Tools,10023,https://www.promptmule.com
2403,Node Code Studio,Apex Solutions,2025,Coding Assistant,Coding,No-code platform for building and automating A...,Subscription,Private Beta,32k tokens,API / Developer Tools,22579,https://nodecodestudio.com


In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   AI_Name            10000 non-null  object
 1   Developer          10000 non-null  object
 2   Release_Year       10000 non-null  int64 
 3   Intelligence_Type  10000 non-null  object
 4   Primary_Domain     10000 non-null  object
 5   Key_Functionality  10000 non-null  object
 6   Pricing_Model      10000 non-null  object
 7   API_Availability   10000 non-null  object
 8   Context_Window     10000 non-null  object
 9   Accessibility      10000 non-null  object
 10  Popularity_Votes   10000 non-null  int64 
 11  Website_URL        10000 non-null  object
dtypes: int64(2), object(10)
memory usage: 937.6+ KB


In [70]:
for col in df.columns:
    print(f"{col} -> {df[col].nunique()}")

AI_Name -> 10000
Developer -> 2889
Release_Year -> 5
Intelligence_Type -> 8
Primary_Domain -> 8
Key_Functionality -> 9968
Pricing_Model -> 6
API_Availability -> 6
Context_Window -> 6
Accessibility -> 6
Popularity_Votes -> 9062
Website_URL -> 9931


In [71]:
df.columns

Index(['AI_Name', 'Developer', 'Release_Year', 'Intelligence_Type',
       'Primary_Domain', 'Key_Functionality', 'Pricing_Model',
       'API_Availability', 'Context_Window', 'Accessibility',
       'Popularity_Votes', 'Website_URL'],
      dtype='object')

In [72]:
df = df.drop(columns=["AI_Name", "Developer", "Website_URL", "Key_Functionality", "Release_Year"])

- This above columns we not used
- We perform model building with this data and predict the popularity vote

In [73]:
df.sample(2)

,Intelligence_Type,Primary_Domain,Pricing_Model,API_Availability,Context_Window,Accessibility,Popularity_Votes
5860,Multimodal Generative AI,General/Other,Open Source,No,1M tokens,Browser Extension,43370
5996,Computer Vision / Generative Art,Image/Design,Free,Yes (GraphQL),1M tokens,API / Developer Tools,21216


Thinking process:
- First we do one hot encoding of all of object columns
- Then train test split and model building
- So let's do 

In [74]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [75]:
X = df.drop(columns=["Popularity_Votes"])
y = df[["Popularity_Votes"]]

In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [77]:
ohe = OneHotEncoder(handle_unknown="ignore")

In [78]:
X_train_ohe = ohe.fit_transform(X_train)
X_test_ohe = ohe.transform(X_test)

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Intelligence_Type  10000 non-null  object
 1   Primary_Domain     10000 non-null  object
 2   Pricing_Model      10000 non-null  object
 3   API_Availability   10000 non-null  object
 4   Context_Window     10000 non-null  object
 5   Accessibility      10000 non-null  object
 6   Popularity_Votes   10000 non-null  int64 
dtypes: int64(1), object(6)
memory usage: 547.0+ KB


In [80]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_ohe, y_train)

y_pred = model.predict(X_test_ohe)

In [81]:
from sklearn.metrics import r2_score, mean_absolute_error

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2: -0.00873200050014633
MAE: 12395.169000890824


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

from sklearn import set_config
set_config(display='diagram')

In [ ]:

categorical_cols = X.select_dtypes(include="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

In [98]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Intelligence_Type', 'Primary_Domain', 'Pricing_Model',
       'API_Availability', 'Context_Window', 'Accessibility'],
      dtype='object'))])),
                ('regressor', LinearRegression())])

In [96]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [97]:
from sklearn.metrics import r2_score, mean_absolute_error

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2: -0.00873200050014633
MAE: 12395.169000890824
